## Step 1. Load SciQ dataset
1, Load the SciQ dataset

2, Print the dataset structure

3, Print one sample from the training set

In [1]:
"""import sys
!{sys.executable} -m pip install --no-cache-dir datasets"""

'import sys\n!{sys.executable} -m pip install --no-cache-dir datasets'

Basic setup

In [2]:
from datasets import load_dataset

# Download the SciQ dataset
dataset = load_dataset("sciq")

# Print the dataset structure
print("Dataset structure:\n", dataset)

# Print one sample from the training set
print("Sample from training set:\n", dataset["train"][0])

c:\Users\S\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset structure:
 DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})
Sample from training set:
 {'question': 'What type of organism is commonly used in preparation of foods such as cheese and yogurt?', 'distractor3': 'viruses', 'distractor1': 'protozoa', 'distractor2': 'gymnosperms', 'correct_answer': 'mesophilic organisms', 'support': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many 

Use unique support paragraphs as documents, and store the gold doc id per question.

In [3]:
documents = []
support_to_id = {}

# Build unique corpus
for ex in dataset["train"]:
    s = ex["support"]
    if s not in support_to_id:
        support_to_id[s] = len(documents)
        documents.append({"id": support_to_id[s], "text": s})

# Gold doc id for each training question
gold_doc_id = [support_to_id[ex["support"]] for ex in dataset["train"]]

print("Unique documents:", len(documents))
print("Example doc:", documents[0])
print("Gold doc id for train[0]:", gold_doc_id[0])

Unique documents: 10474
Example doc: {'id': 0, 'text': 'Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.'}
Gold doc id for train[0]: 0


Build embeddings + FAISS index

In [4]:
"""import sys
!{sys.executable} -m pip install --no-cache-dir sentence-transformers faiss-cpu"""

'import sys\n!{sys.executable} -m pip install --no-cache-dir sentence-transformers faiss-cpu'

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2") # if GPU is available, add this: device="cuda"

texts = [doc["text"] for doc in documents]
embeddings = model.encode(texts, convert_to_numpy=True)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print("Index size:", index.ntotal)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1335.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index size: 10474


#### Retriever baseline

compute Recall@1 / @3 / @5 for your SciQ retriever (before any LLM)

build corpus (dedup), build FAISS, compute Recall@k

Embedding: all-MiniLM-L6-v2, FAISS IndexFlatL2, no chunking, document-level retrieval.

These are the baseline retrieval metrics

We now treat supports as an external KB, so we build the index from all supports (train+validation+test)!

In [6]:
import numpy as np

# 1) Build unique document corpus + mapping (support_text -> doc_id)
documents = []
support_to_id = {}

for split in ["train", "validation", "test"]:
    for ex in dataset[split]:
        s = ex["support"]
        if s not in support_to_id:
            support_to_id[s] = len(documents)
            documents.append({"id": support_to_id[s], "text": s})

print("Unique documents (all splits):", len(documents))

# 2) Embed documents + build FAISS index
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_texts = [d["text"] for d in documents]
doc_emb = model.encode(doc_texts, convert_to_numpy=True, show_progress_bar=True)

index = faiss.IndexFlatL2(doc_emb.shape[1])
index.add(doc_emb)

print("Index size:", index.ntotal)

# 3) Recall@k evaluation
def recall_at_k(split="validation", k=3, n=500, seed=0):
    """
    Train-only index is assumed:
      - documents/support_to_id/index were built from dataset['train'] supports only.

    Returns:
      coverage: fraction of eval questions whose gold support exists in train-only corpus
      recall_covered: recall@k computed only on covered questions
      recall_overall: recall@k over all questions (missing-gold counts as 0)
    """
    data_full = dataset[split]

    # Fixed random subset for fair comparisons across settings
    if n is None:
        data = data_full
    else:
        rng = np.random.default_rng(seed)
        idxs = rng.choice(len(data_full), size=min(n, len(data_full)), replace=False)
        data = data_full.select(sorted(idxs.tolist()))

    covered = 0
    correct_covered = 0
    total = len(data)

    for ex in data:
        q = ex["question"]
        gold_support = ex["support"]

        # Retrieve
        q_emb = model.encode([q], convert_to_numpy=True).astype(np.float32)
        _, retrieved = index.search(q_emb, k)  # retrieved shape (1, k)

        # If gold support isn't in the train-only corpus, we can't retrieve it
        if gold_support in support_to_id:
            covered += 1
            gold_id = support_to_id[gold_support]
            if gold_id in retrieved[0]:
                correct_covered += 1

    coverage = covered / total if total else 0.0
    recall_covered = (correct_covered / covered) if covered else 0.0
    recall_overall = (correct_covered / total) if total else 0.0

    return coverage, recall_covered, recall_overall

"""for k in [1, 3, 5]:
    cov, r_cov, r_all = recall_at_k(split="train", k=k, n=500, seed=0)
    print(f"[train] k={k}  coverage={cov:.3f}  recall(covered)={r_cov:.3f}  recall(overall)={r_all:.3f}")"""

for k in [1, 3, 5]:
    cov, r_cov, r_all = recall_at_k(split="validation", k=k, n=500, seed=0)
    print(f"[validation] k={k}  coverage={cov:.3f}  recall(covered)={r_cov:.3f}  recall(overall)={r_all:.3f}")

"""this part only for final report
for k in [1, 3, 5]:
    cov, r_cov, r_all = recall_at_k(split="test", k=k, n=500, seed=0)
    print(f"[test] k={k}  coverage={cov:.3f}  recall(covered)={r_cov:.3f}  recall(overall)={r_all:.3f}")"""

Unique documents (all splits): 12242


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1448.06it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 383/383 [00:35<00:00, 10.67it/s]


Index size: 12242
[validation] k=1  coverage=1.000  recall(covered)=0.450  recall(overall)=0.450
[validation] k=3  coverage=1.000  recall(covered)=0.620  recall(overall)=0.620
[validation] k=5  coverage=1.000  recall(covered)=0.678  recall(overall)=0.678


'this part only for final report\nfor k in [1, 3, 5]:\n    cov, r_cov, r_all = recall_at_k(split="test", k=k, n=500, seed=0)\n    print(f"[test] k={k}  coverage={cov:.3f}  recall(covered)={r_cov:.3f}  recall(overall)={r_all:.3f}")'

build MCQ options + correct letter

In [7]:
import random

def make_mcq(example, seed=0):
    rnd = random.Random(seed)

    # SciQ format: three separate distractor fields
    choices = [
        example["distractor1"],
        example["distractor2"],
        example["distractor3"],
        example["correct_answer"],
    ]

    rnd.shuffle(choices)

    correct_idx = choices.index(example["correct_answer"])
    correct_letter = "ABCD"[correct_idx]

    return example["question"], choices, correct_letter


def format_prompt(question, choices):
    return (
        "Answer the multiple-choice question.\n"
        "Return ONLY valid JSON with the following format:\n"
        '{"answer": "A"}\n'
        'where "answer" must be one of "A", "B", "C", "D".\n'
        "Do not output any other text.\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
    )


def extract_letter(text):
    text = text.strip().upper()
    for ch in text:
        if ch in ["A", "B", "C", "D"]:
            return ch
    return None

LLM-only baseline with a free Gemini model

In [8]:
import sys
!{sys.executable} -m pip install -q -U google-genai

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# list available models
from google import genai
import os

os.environ["GEMINI_API_KEY"] = "xxx"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

models = list(client.models.list())
print("Total:", len(models))
for m in models[:50]:
    name = getattr(m, "name", None) or str(m)
    methods = getattr(m, "supported_actions", None) or getattr(m, "supported_methods", None)
    print(name, methods)

In [10]:
from google import genai
from google.genai import errors
import os, json, time, random, re
from collections import deque

class RateLimiter:
    def __init__(self, rpm: int):
        self.rpm = rpm
        self.calls = deque()  # timestamps

    def wait(self):
        now = time.time()
        window = 60.0
        # 清掉窗口外的调用
        while self.calls and now - self.calls[0] > window:
            self.calls.popleft()

        if len(self.calls) >= self.rpm:
            sleep_s = window - (now - self.calls[0]) + 0.2  # 0.2s buffer
            time.sleep(max(0.0, sleep_s))

        self.calls.append(time.time())

limiter = RateLimiter(rpm=900)

# NOTE: Avoid hardcoding real keys in notebooks. Use an env var instead.
# os.environ["GEMINI_API_KEY"] should already be set outside the notebook if possible.
os.environ["GEMINI_API_KEY"] = "xxx"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Use a model name that actually exists in your ListModels output (must include 'models/')
MODEL_NAME = "models/gemini-2.5-flash"  # or "models/gemini-2.5-flash"

MCQ_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string", "enum": ["A", "B", "C", "D"]}
    },
    "required": ["answer"],
    "additionalProperties": False,
}

def _strip_code_fences(text: str) -> str:
    """Remove ```json ... ``` wrappers if the model returns fenced JSON."""
    if not text:
        return ""
    t = text.strip()
    if t.startswith("```"):
        # Remove triple backticks and optional language tag like ```json
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
        t = re.sub(r"\s*```$", "", t)
    return t.strip()

def llm_answer_letter(prompt: str, max_retries: int = 3) -> str | None:
    """
    Calls the model and returns one of 'A','B','C','D' or None.
    Retries transient failures (429/5xx) with exponential backoff.
    """
    last_err = None

    for attempt in range(max_retries):
        try:
            limiter.wait()  # limit rate before each request
            resp = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config={
                    "temperature": 0.0,
                    "response_mime_type": "application/json",
                    "response_json_schema": MCQ_SCHEMA,
                },
            )

            txt = _strip_code_fences(resp.text or "")
            data = json.loads(txt)
            ans = data.get("answer")
            return ans if ans in ("A", "B", "C", "D") else None

        except (errors.ServerError, errors.ClientError) as e:
            
            code = getattr(e, "status_code", None)
            print("ERROR TYPE:", type(e))
            print("ERROR REPR:", repr(e))

            # Retry on rate limits and transient server errors
            if code in (429, 500, 502, 503, 504) or code is None:
                sleep_s = min(20.0, (2 ** attempt) + random.random())
                print(f"  retry {attempt+1}/{max_retries} (status={code}), sleeping {sleep_s:.2f}s")
                time.sleep(sleep_s)
                continue
            # For other 4xx errors (bad request, auth, etc.), fail fast
            raise 

            

        except (json.JSONDecodeError, TypeError, KeyError):
            # If JSON parsing fails, treat as unparsed
            return None
        except Exception as e:
            # Catch network/SSL/proxy/timeouts that are not ServerError/ClientError
            print("UNEXPECTED ERROR TYPE:", type(e))
            print("UNEXPECTED ERROR REPR:", repr(e))
            return None

    # If we exhaust retries, return None and let evaluation continue
    return None

def eval_llm_only(split: str = "validation", n: int = 50, seed: int = 0):
    """
    Evaluates LLM accuracy on N examples.
    'bad' counts cases where the output couldn't be parsed into A/B/C/D.
    'throttle_s' adds a small delay between calls to reduce rate limiting.
    """
    data = dataset[split].select(range(min(n, len(dataset[split]))))

    correct = 0
    total = 0
    bad = 0

    for i, ex in enumerate(data):
        t1 = time.time()
        q, choices, gold = make_mcq(ex, seed=seed + i)  # assumed defined elsewhere
        prompt = format_prompt(q, choices)              # assumed defined elsewhere

        pred = llm_answer_letter(prompt)
        print(f"{i+1}/{len(data)}  took {time.time()-t1:.2f}s  pred={pred}  gold={gold}")
        total += 1

        if pred is None:
            bad += 1
        elif pred == gold:
            correct += 1

    acc = correct / total if total else 0.0
    return acc, bad, total

#LLM-only run

"""acc, bad, total = eval_llm_only(split="validation", n=200)
print(f"LLM-only accuracy: {acc:.4f}  (unparsed={bad}/{total})")"""

'acc, bad, total = eval_llm_only(split="validation", n=200)\nprint(f"LLM-only accuracy: {acc:.4f}  (unparsed={bad}/{total})")'

chunking part start here!

In [11]:
from transformers import AutoTokenizer
import numpy as np
import faiss

# fix chunking tokenizer
CHUNK_TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
chunk_tokenizer = AutoTokenizer.from_pretrained(CHUNK_TOKENIZER_NAME, use_fast=True)

def chunk_text_tokens(text: str, chunk_tokens: int, overlap_tokens: int = 0, tokenizer=None) -> list[str]:
    tokenizer = tokenizer or chunk_tokenizer
    assert chunk_tokens > 0
    assert 0 <= overlap_tokens < chunk_tokens

    ids = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    stride = chunk_tokens - overlap_tokens

    for start in range(0, len(ids), stride):
        end = min(len(ids), start + chunk_tokens)
        chunk_ids = ids[start:end]
        if not chunk_ids:
            break
        chunk = tokenizer.decode(chunk_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunks.append(chunk)
        if end == len(ids):
            break
    return chunks

def build_chunk_corpus(chunk_tokens: int, overlap_tokens: int, tokenizer=None):
    tokenizer = tokenizer or chunk_tokenizer
    chunk_docs = []
    chunk_to_source_support = []  # chunk_id -> original support text

    unique_supports = list(support_to_id.keys())
    for support in unique_supports:
        chunks = chunk_text_tokens(support, chunk_tokens=chunk_tokens, overlap_tokens=overlap_tokens, tokenizer=tokenizer)
        for ch in chunks:
            chunk_docs.append(ch)
            chunk_to_source_support.append(support)
    return chunk_docs, chunk_to_source_support

def build_faiss_for_chunks(chunk_docs, embed_model):
    emb = embed_model.encode(chunk_docs, convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    idx = faiss.IndexFlatL2(emb.shape[1])
    idx.add(emb)
    return idx

def encode_q_cached(q: str, embed_model, cache: dict):
    if q in cache:
        return cache[q]
    emb = embed_model.encode([q], convert_to_numpy=True).astype(np.float32)  # (1, dim)
    cache[q] = emb
    return emb

def recall_at_k_chunks(data_subset, k: int, chunk_index, chunk_to_source, embed_model, q_cache: dict):
    correct = 0
    total = len(data_subset)

    for ex in data_subset:
        q = ex["question"]
        gold_support = ex["support"]

        q_emb = encode_q_cached(q, embed_model, q_cache)
        _, retrieved = chunk_index.search(q_emb, k)

        hit = any(chunk_to_source[i] == gold_support for i in retrieved[0])
        if hit:
            correct += 1

    return correct / total if total else 0.0

In [12]:
import numpy as np

def make_fixed_subset(split="validation", n=200, seed=0):
    data_full = dataset[split]
    if n is None or n >= len(data_full):
        return data_full
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(data_full), size=n, replace=False)
    return data_full.select(sorted(idxs.tolist()))

# subset for all experiment
VAL_N = 200
SEED = 0
val_subset = make_fixed_subset("validation", n=VAL_N, seed=SEED)

In [13]:
def retrieve_chunks(question, k, chunk_index, chunk_docs, embed_model):
    q_emb = embed_model.encode([question], convert_to_numpy=True).astype(np.float32)
    _, idxs = chunk_index.search(q_emb, k)  # idxs shape (1, k)
    return [chunk_docs[i] for i in idxs[0]]

def eval_rag(data_subset, k, chunk_index, chunk_docs, embed_model, llm_answer_fn):
    """
    llm_answer_fn(prompt) -> "A"/"B"/"C"/"D" or None
    return (acc, bad, total)
    """
    correct = 0
    bad = 0
    total = len(data_subset)

    for ex in data_subset:
        q = ex["question"]
        question, choices, gold_letter = make_mcq(ex, seed=0)
        ctxs = retrieve_chunks(q, k, chunk_index, chunk_docs, embed_model)
        prompt = format_rag_prompt(ctxs, q, choices)

        pred = llm_answer_fn(prompt)
        if pred is None:
            bad += 1
            continue
        if pred == gold_letter:
            correct += 1

    denom = max(1, (total - bad))
    return correct / denom, bad, total

def format_rag_prompt(context_passages, question, choices):
    context = "\n\n".join([f"[Context {i+1}]\n{p}" for i, p in enumerate(context_passages)])
    return (
        "Use the context to answer the multiple-choice question.\n"
        "Return ONLY valid JSON in exactly this format:\n"
        '{"answer": "A"}\n'
        'where "answer" must be one of "A", "B", "C", "D".\n'
        "Do not output any explanation or extra text.\n\n"
        f"{context}\n\n"
        f"Question: {question}\n"
        f"A) {choices[0]}\n"
        f"B) {choices[1]}\n"
        f"C) {choices[2]}\n"
        f"D) {choices[3]}\n"
    )


In [14]:
VAL_N = 200
SEED = 0
val_subset = make_fixed_subset("validation", n=VAL_N, seed=SEED)

CHUNK_SETTINGS = [
    (256, 64),
    (512, 128),
    (1024, 256),
]

KS = [1, 3, 5]

EMBED_MODELS = [
    ("all-MiniLM-L6-v2", "sentence-transformers/all-MiniLM-L6-v2"),
    ("msmarco-MiniLM-L6-v3", "sentence-transformers/msmarco-MiniLM-L6-v3"),
]

llm_answer_fn = llm_answer_letter

chunk-level Recall@k only

no LLM

In [15]:
import pandas as pd
from sentence_transformers import SentenceTransformer

rows = []

for embed_short, embed_hf_name in EMBED_MODELS:
    embed_model = SentenceTransformer(embed_hf_name)
    q_cache = {}

    for chunk_tokens, overlap_tokens in CHUNK_SETTINGS:
        chunk_docs, chunk_to_source = build_chunk_corpus(
            chunk_tokens, overlap_tokens, tokenizer=chunk_tokenizer
        )
        chunk_index = build_faiss_for_chunks(chunk_docs, embed_model)

        for k in KS:
            r = recall_at_k_chunks(
                data_subset=val_subset,
                k=k,
                chunk_index=chunk_index,
                chunk_to_source=chunk_to_source,
                embed_model=embed_model,
                q_cache=q_cache,
            )
            rows.append({
                "stage": "recall",
                "embed_model": embed_short,
                "chunk_tokens": chunk_tokens,
                "overlap_tokens": overlap_tokens,
                "k": k,
                "metric": f"recall@{k}",
                "value": r
            })

df_recall = pd.DataFrame(rows)
display(df_recall)

pivot_recall = df_recall.pivot_table(
    index=["embed_model", "chunk_tokens", "overlap_tokens"],
    columns=["metric"],
    values="value",
    aggfunc="mean"
).reset_index()

display(pivot_recall)
pivot_recall.to_csv("chunk_recall_summary.csv", index=False)
print("Saved: chunk_recall_summary.csv")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1475.61it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Token indices sequence length is longer than the specified maximum sequence length for this model (626 > 512). Running this sequence through the model will result in indexing errors
Batches: 100%|██████████| 383/383 [00:43<00:00,  8.88it/s]
c:\Users\S\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\S\.cache\huggingface\hub\models--sente

,stage,embed_model,chunk_tokens,overlap_tokens,k,metric,value
0,recall,all-MiniLM-L6-v2,256,64,1,recall@1,0.485
1,recall,all-MiniLM-L6-v2,256,64,3,recall@3,0.610
2,recall,all-MiniLM-L6-v2,256,64,5,recall@5,0.680
3,recall,all-MiniLM-L6-v2,512,128,1,recall@1,0.485
4,recall,all-MiniLM-L6-v2,512,128,3,recall@3,0.615
5,recall,all-MiniLM-L6-v2,512,128,5,recall@5,0.680
6,recall,all-MiniLM-L6-v2,1024,256,1,recall@1,0.485
7,recall,all-MiniLM-L6-v2,1024,256,3,recall@3,0.615
8,recall,all-MiniLM-L6-v2,1024,256,5,recall@5,0.680
9,recall,msmarco-MiniLM-L6-v3,256,64,1,recall@1,0.500


metric,embed_model,chunk_tokens,overlap_tokens,recall@1,recall@3,recall@5
0,all-MiniLM-L6-v2,256,64,0.485,0.610,0.680
1,all-MiniLM-L6-v2,512,128,0.485,0.615,0.680
2,all-MiniLM-L6-v2,1024,256,0.485,0.615,0.680
3,msmarco-MiniLM-L6-v3,256,64,0.500,0.620,0.660
4,msmarco-MiniLM-L6-v3,512,128,0.500,0.630,0.675
5,msmarco-MiniLM-L6-v3,1024,256,0.500,0.630,0.675


Saved: chunk_recall_summary.csv


RAG accuracy

In [ ]:
# Sanity check：n=50 + k=3
VAL_N_RAG = 50
val_subset_rag = make_fixed_subset("validation", n=VAL_N_RAG, seed=SEED)

KS_RAG = [3]

rows = []

for embed_short, embed_hf_name in EMBED_MODELS:
    embed_model = SentenceTransformer(embed_hf_name)

    for chunk_tokens, overlap_tokens in CHUNK_SETTINGS:
        chunk_docs, chunk_to_source = build_chunk_corpus(
            chunk_tokens, overlap_tokens, tokenizer=chunk_tokenizer
        )
        chunk_index = build_faiss_for_chunks(chunk_docs, embed_model)

        for k in KS_RAG:
            acc, bad, total = eval_rag(
                data_subset=val_subset_rag,
                k=k,
                chunk_index=chunk_index,
                chunk_docs=chunk_docs,
                embed_model=embed_model,
                llm_answer_fn=llm_answer_fn
            )
            rows.append({
                "stage": "rag_sanity",
                "embed_model": embed_short,
                "chunk_tokens": chunk_tokens,
                "overlap_tokens": overlap_tokens,
                "k": k,
                "metric": f"rag_acc@{k}",
                "value": acc,
                "bad": bad,
                "total": total
            })

df_rag_sanity = pd.DataFrame(rows)
display(df_rag_sanity)

RAG accuracy final test

In [ ]:
# final run: n=200 + k=1/3/5
VAL_N_RAG = 200
val_subset_rag = make_fixed_subset("validation", n=VAL_N_RAG, seed=SEED)
KS_RAG = [1, 3, 5]

rows = []

for embed_short, embed_hf_name in EMBED_MODELS:
    embed_model = SentenceTransformer(embed_hf_name)

    for chunk_tokens, overlap_tokens in CHUNK_SETTINGS:
        chunk_docs, chunk_to_source = build_chunk_corpus(
            chunk_tokens, overlap_tokens, tokenizer=chunk_tokenizer
        )
        chunk_index = build_faiss_for_chunks(chunk_docs, embed_model)

        for k in KS_RAG:
            acc, bad, total = eval_rag(
                data_subset=val_subset_rag,
                k=k,
                chunk_index=chunk_index,
                chunk_docs=chunk_docs,
                embed_model=embed_model,
                llm_answer_fn=llm_answer_fn
            )
            rows.append({
                "stage": "rag",
                "embed_model": embed_short,
                "chunk_tokens": chunk_tokens,
                "overlap_tokens": overlap_tokens,
                "k": k,
                "metric": f"rag_acc@{k}",
                "value": acc,
                "bad": bad,
                "total": total
            })

df_rag = pd.DataFrame(rows)
display(df_rag)

pivot_rag = df_rag.pivot_table(
    index=["embed_model", "chunk_tokens", "overlap_tokens"],
    columns=["metric"],
    values="value",
    aggfunc="mean"
).reset_index()

display(pivot_rag)
pivot_rag.to_csv("rag_accuracy_summary.csv", index=False)
print("Saved: rag_accuracy_summary.csv")